# 🇻🇳 Vietnamese Fake News: Crawling & Preprocessing Pipeline

**Complete workflow for Vietnamese news data collection and processing**


In [ ]:
# Install dependencies
import subprocess
import sys

packages = [
    "pandas",
    "numpy",
    "torch",
    "torchvision",
    "transformers",
    "pillow",
    "matplotlib",
    "datasets",
    "huggingface_hub",
    "loguru",
    "tqdm",
    "beautifulsoup4",
    "lxml",
    "httpx",
    "nest-asyncio",
]

for pkg in packages:
    try:
        __import__(pkg.replace("-", "_"))
        print(f"✅ {pkg}")
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
        print(f"📦 Installed {pkg}")

: 

In [ ]:
# Setup
import sys, os, json, asyncio, nest_asyncio
import pandas as pd, numpy as np, torch
from pathlib import Path
from typing import List, Dict, Tuple
from datetime import datetime

nest_asyncio.apply()
os.environ["OPENSSL_CONF"] = "openssl.cnf"

project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "src"))

print(f"✅ Ready: {project_root}")

: 

In [ ]:
# Import crawler modules
try:
    from src.crawler.crawler_factory import CrawlerFactory
    from src.processing.dataset_handler import DatasetHandler
    from src.helpers.logger import logger

    print("✅ Crawler modules imported")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Install missing: pip install loguru tqdm beautifulsoup4 lxml httpx")

In [ ]:
# Configuration
CRAWLING_CONFIG = {
    "dataset_name": "tranthaihoa/vifactcheck",
    "url_column": "Url",
    "splits": ["train", "test", "dev"],
    "url_limit": None,  # ALL URLs
    "cache_dir": "./data/caches",
    "output_dir": "./src/data/json",
}

os.makedirs(CRAWLING_CONFIG["cache_dir"], exist_ok=True)
os.makedirs(CRAWLING_CONFIG["output_dir"], exist_ok=True)

print("🔧 Configuration ready")

In [ ]:
async def crawl_dataset():
    """Crawl VifactCheck dataset"""
    results = []

    for split in CRAWLING_CONFIG["splits"]:
        print(f"\n--- Processing {split} ---")

        crawler_factory = CrawlerFactory(
            cache_filename=f"{CRAWLING_CONFIG['cache_dir']}/crawling_status_{split}.json",
            failed_log_filename=f"{CRAWLING_CONFIG['cache_dir']}/failed_urls_{split}.json",
        )

        if crawler_factory.check_cache_file_exists():
            crawler_factory.clear_cache()

        dataset_handler = DatasetHandler(CRAWLING_CONFIG["dataset_name"])
        urls = dataset_handler.get_urls_from_split(
            split=split,
            url_column=CRAWLING_CONFIG["url_column"],
            limit=CRAWLING_CONFIG["url_limit"],
        )

        if urls:
            print(f"Found {len(urls)} URLs")
            output_filename = f"news_data_vifactcheck_{split}.json"
            await crawler_factory.crawl_and_save_all(
                urls, output_filename, format_name="custom"
            )
            results.append({"split": split, "count": len(urls)})
            print(f"✅ {split} completed")

    return results


print("🔧 Crawl function ready")

In [ ]:
# Execute crawling (uncomment to run)
# results = await crawl_dataset()
# print(f"✅ Crawled {len(results)} splits")

print("🚀 Ready to crawl ALL URLs")
print("💡 Uncomment the line above to start")

In [ ]:
# Preprocessing setup
PREPROCESSING_CONFIG = {
    "text_model": "vinai/phobert-base",
    "image_model": "resnet18",
    "language": "vi",
    "max_length": 512,
    "batch_size": 16,
    "save_format": "pkl",
}

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"🔧 Device: {device}")

In [ ]:
# Initialize preprocessor
try:
    from src.preprocessing import CombinedPreprocessor

    preprocessor = CombinedPreprocessor(
        text_model_name=PREPROCESSING_CONFIG["text_model"],
        image_model_name=PREPROCESSING_CONFIG["image_model"],
        language=PREPROCESSING_CONFIG["language"],
        max_text_length=PREPROCESSING_CONFIG["max_length"],
        device=device,
    )
    print("✅ Preprocessor ready")
except Exception as e:
    print(f"❌ Preprocessor error: {e}")

In [ ]:
# Load crawled data
def load_data():
    data_dir = Path(CRAWLING_CONFIG["output_dir"])
    if not data_dir.exists():
        return []

    articles = []
    for json_file in data_dir.glob("news_data_vifactcheck_*.json"):
        with open(json_file, "r", encoding="utf-8") as f:
            data = json.load(f)
            articles.extend(data)

    return articles


crawled_articles = load_data()
print(f"📊 Loaded {len(crawled_articles)} articles")

In [ ]:
# Prepare data
if crawled_articles:
    texts = []
    image_paths = []
    labels = []

    for article in crawled_articles:
        text = (
            article.get("text", "")
            or article.get("content", "")
            or article.get("title", "")
        )
        if text.strip():
            texts.append(text.strip())
            images = article.get("images", [])
            image_paths.append(images[0] if images else None)
            labels.append(article.get("label", 0))

    print(f"📊 Prepared {len(texts)} samples")
else:
    print("❌ No data available")

In [ ]:
# Create placeholders
if "texts" in locals() and texts:
    from PIL import Image

    placeholder_dir = Path("./placeholder_images")
    placeholder_dir.mkdir(exist_ok=True)

    processed_paths = []
    for i, img_path in enumerate(image_paths):
        if not img_path:
            placeholder_path = placeholder_dir / f"placeholder_{i}.jpg"
            if not placeholder_path.exists():
                placeholder_array = np.random.randint(
                    128, 200, (224, 224, 3), dtype=np.uint8
                )
                Image.fromarray(placeholder_array).save(placeholder_path)
            processed_paths.append(str(placeholder_path))
        else:
            processed_paths.append(img_path)

    print(f"✅ Created {len(processed_paths)} image paths")

In [ ]:
# Batch preprocessing to prevent memory issues
if "preprocessor" in locals() and "texts" in locals() and texts:
    try:
        import gc
        from tqdm import tqdm

        os.makedirs("./processed_data/crawled", exist_ok=True)

        # Configuration for batch processing
        BATCH_SIZE = 32  # Process 32 samples at a time
        total_samples = len(texts)
        num_batches = (total_samples + BATCH_SIZE - 1) // BATCH_SIZE

        print(f"📊 Total samples: {total_samples}")
        print(f"📦 Batch size: {BATCH_SIZE}")
        print(f"🔢 Number of batches: {num_batches}")

        all_text_features = []
        all_image_features = []
        all_labels = []

        # Process in batches
        for batch_idx in tqdm(range(num_batches), desc="Processing batches"):
            start_idx = batch_idx * BATCH_SIZE
            end_idx = min((batch_idx + 1) * BATCH_SIZE, total_samples)

            # Get batch data
            batch_texts = texts[start_idx:end_idx]
            batch_image_paths = processed_paths[start_idx:end_idx]
            batch_labels = labels[start_idx:end_idx]

            # Process text features for this batch
            text_features = preprocessor.text_preprocessor.extract_token_embeddings(
                batch_texts
            )

            # Process image features for this batch
            image_features = preprocessor.image_preprocessor.extract_features(
                batch_image_paths
            )

            # Collect features
            all_text_features.append(text_features)
            all_image_features.append(image_features)
            all_labels.extend(batch_labels)

            # Clear memory after each batch
            del text_features, image_features
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            elif torch.backends.mps.is_available():
                torch.mps.empty_cache()

        # Concatenate all features
        print("🔄 Concatenating features...")
        text_features_all = np.vstack(all_text_features)
        image_features_all = np.vstack(all_image_features)
        labels_all = np.array(all_labels)

        # Save features
        save_format = PREPROCESSING_CONFIG["save_format"]
        save_dir = "./processed_data/crawled"

        # Save text features
        text_save_path = os.path.join(save_dir, f"text_features.{save_format}")
        if save_format == "pkl":
            import pickle

            with open(text_save_path, "wb") as f:
                pickle.dump({"features": text_features_all, "labels": labels_all}, f)
        else:  # npz
            np.savez(text_save_path, features=text_features_all, labels=labels_all)

        # Save image features
        image_save_path = os.path.join(save_dir, f"image_features.{save_format}")
        if save_format == "pkl":
            with open(image_save_path, "wb") as f:
                pickle.dump({"features": image_features_all, "labels": labels_all}, f)
        else:  # npz
            np.savez(image_save_path, features=image_features_all, labels=labels_all)

        # Save combined dataset
        combined_save_path = os.path.join(save_dir, f"combined_dataset.{save_format}")
        if save_format == "pkl":
            with open(combined_save_path, "wb") as f:
                pickle.dump(
                    {
                        "text_features": text_features_all,
                        "image_features": image_features_all,
                        "labels": labels_all,
                    },
                    f,
                )
        else:  # npz
            np.savez(
                combined_save_path,
                text_features=text_features_all,
                image_features=image_features_all,
                labels=labels_all,
            )

        # Create and save metadata
        metadata = {
            "num_samples": total_samples,
            "text_feature_shape": text_features_all.shape,
            "image_feature_shape": image_features_all.shape,
            "num_classes": len(set(labels)),
            "language": preprocessor.language,
            "text_model": preprocessor.text_preprocessor.model_name,
            "image_model": preprocessor.image_preprocessor.model_name,
            "save_dir": save_dir,
            "batch_size": BATCH_SIZE,
            "num_batches": num_batches,
            "files": {
                "text_features": f"text_features.{save_format}",
                "image_features": f"image_features.{save_format}",
                "combined": f"combined_dataset.{save_format}",
            },
        }

        import json

        with open(os.path.join(save_dir, "metadata.json"), "w") as f:
            json.dump(metadata, f, indent=2)

        print("✅ Batch preprocessing complete!")
        print(f"📊 Total samples processed: {total_samples}")
        print(f"📝 Text features shape: {text_features_all.shape}")
        print(f"🖼️ Image features shape: {image_features_all.shape}")
        print(f"💾 Saved to: {save_dir}")

        # Clear memory
        del all_text_features, all_image_features, text_features_all, image_features_all
        gc.collect()

    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback

        traceback.print_exc()
else:
    print("❌ Cannot preprocess - missing data or preprocessor")

In [ ]:
# Status check
def check_status():
    print("🚀 Pipeline Status:")

    data_dir = Path(CRAWLING_CONFIG["output_dir"])
    if data_dir.exists():
        json_files = list(data_dir.glob("news_data_vifactcheck_*.json"))
        print(f"📰 Crawled: {len(json_files)} files")

    processed_dir = Path("./processed_data/crawled")
    if processed_dir.exists():
        files = list(processed_dir.glob("*"))
        print(f"🔄 Processed: {len(files)} files")

    print("\n🎯 Next steps:")
    print("1. Run crawling (uncomment line)")
    print("2. Execute preprocessing")
    print("3. Analyze results")


check_status()